# 01 Data audit and EDA
Point-in-time audit, leakage checks and split justification for the no-show triage project.

Make the project folder importable.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

Import libraries and project modules.

In [ ]:
from src import config, data, evaluate
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

Load the raw file and show its shape and column types.

In [ ]:
raw = data.load_raw()
print(raw.shape)
display(raw.dtypes.to_frame("dtype"))
display(raw.head())

Record the data hash, file size and row count for data/README.md.

In [ ]:
md5 = data.file_md5()
size_mb = config.DATA_RAW.stat().st_size / 1e6
print("file:", config.DATA_RAW.name)
print("md5:", md5)
print("size MB:", round(size_mb, 2))
print("rows:", len(raw), "columns:", raw.shape[1])

Show the point-in-time audit of every column.

In [ ]:
audit = pd.DataFrame(config.COLUMN_AUDIT)
with pd.option_context("display.max_colwidth", 80):
    display(audit)

Count missing values per column in the raw file.

In [ ]:
display(raw.isna().sum().to_frame("missing"))

Check identifier quality: unique patients, unique appointments and truncated-looking patient ids.

In [ ]:
pid = pd.to_numeric(raw["patient_id"]).round().astype("int64")
print("rows:", len(raw))
print("unique appointment ids:", raw["appointment_id"].nunique())
print("unique patients:", pid.nunique())
print("share of patient ids that are multiples of 1,000,000:",
      round(float((pid % 10**6 == 0).mean()), 4))

Inspect invalid ages before cleaning.

In [ ]:
print(raw["age"].describe())
print("negative ages:", int((raw["age"] < 0).sum()))
print("ages above", config.AGE_MAX, ":", int(
    (raw["age"] > config.AGE_MAX).sum()))

Clean the data and report every row removed and why.

In [ ]:
cleaned, clean_report = data.clean(raw)
display(pd.Series(clean_report).to_frame("rows"))

Build the model table with history features, the 17:00 eligibility rule and the time split.

In [ ]:
table, report = data.build_model_table()
report["data_md5"] = md5
display(pd.Series(report).to_frame("value"))

Compare appointments booked after the 17:00 decision point with the scored population.

In [ ]:
train = table[table["split"] == "train"].copy()
share_out = report["rows_not_booked_by_decision_time"] / report["rows_clean"]
print("share of clean rows booked after the decision point:", round(share_out, 3))
print("their no-show rate:",
      round(report["no_show_rate_not_booked_by_decision_time"], 3))
print("no-show rate of the scored population (train split):",
      round(float(train["no_show"].mean()), 3))

Show class balance on the training split.

In [ ]:
display(train["no_show"].value_counts(
    normalize=True).rename("share").to_frame())

Show split sizes and date ranges without looking at test labels.

In [ ]:
display(table.groupby("split").agg(rows=("no_show", "size"), first_day=(
    "appointment_day", "min"), last_day=("appointment_day", "max")))

Plot the age and lead-time distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(table["age"], bins=40)
axes[0].set_title("Age")
axes[1].hist(table["lead_days"].clip(upper=90), bins=45)
axes[1].set_title("Lead time in days (clipped at 90)")
fig.tight_layout()
plt.show()

Plot no-show rate by lead-time band on the training split.

In [ ]:
banded = evaluate.add_slice_columns(train)
order = ["1 day", "2-7 days", "8-14 days", "15+ days"]
by_lead = banded.groupby("lead_band")["no_show"].agg(
    ["mean", "size"]).reindex(order)
display(by_lead)
by_lead["mean"].plot.bar(title="No-show rate by lead time (train)", rot=0)
plt.show()

Plot no-show rate by age band and appointment weekday on the training split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
age_order = ["0-17", "18-34", "35-49", "50-64", "65+"]
banded.groupby("age_band")["no_show"].mean().reindex(
    age_order).plot.bar(ax=axes[0], title="By age band (train)", rot=0)
banded.groupby("appointment_weekday")["no_show"].mean().plot.bar(
    ax=axes[1], title="By appointment weekday, 0 = Monday (train)", rot=0)
fig.tight_layout()
plt.show()

Measure how much prior-appointment history exists and whether it relates to no-shows.

In [ ]:
print("share of appointments with any prior history:",
      round(float((train["prior_appts"] > 0).mean()), 3))
display(train.groupby("has_history")["no_show"].agg(["mean", "size"]))
display(train["prior_appts"].describe().to_frame("prior_appts"))

Scan each candidate column alone for suspiciously strong prediction (leakage check).

In [ ]:
from sklearn.metrics import roc_auc_score

candidates = config.NUMERIC_FEATURES + ["sms_received", "handcap"]
rows = []
for col in candidates:
    values = train[col].astype(float)
    values = values.fillna(values.median())
    auc = roc_auc_score(train["no_show"], values)
    rows.append({"column": col, "single_column_auc": round(auc, 4),
                "distance_from_chance": round(abs(auc - 0.5), 4)})
scan = pd.DataFrame(rows).sort_values(
    "distance_from_chance", ascending=False).reset_index(drop=True)
display(scan)
print("columns with single-column AUC above 0.80 or below 0.20:",
      scan.loc[scan["distance_from_chance"] > 0.30, "column"].tolist())

Show why sms_received is excluded: how it moves with lead time next to the no-show rate.

In [ ]:
display(banded.groupby("lead_band")[
        ["sms_received", "no_show"]].mean().reindex(order))

Save the processed model table and cleaning report.

In [ ]:
data.save_processed(table, report)
print(sorted(p.name for p in config.DATA_PROCESSED_DIR.iterdir()))

## Findings
Write 4 to 6 sentences here after running: the leakage verdict from the scan, the sms_received decision, why the time split fits deployment, and the data-quality issues removed.